In [1]:
import sys

sys.path.insert(0, "..")
from runner.utils import (
    allocate_benchmarks,
    create_benchmark_campaign,
    load_benchmark_metadata,
)

In [2]:
# If a util function was modified, use this cell to reload it without having to restart the kernel
%run ../runner/utils.py

# 20260415 Run pypsa-de-elec-2-1h (Mosek test)

In [3]:
benchmarks_df = load_benchmark_metadata()
df = benchmarks_df[benchmarks_df["Benchmark"] == "pypsa-de-elec"].copy()

benchs_to_run = df[df["Instance"] == "2-1h"]

In [4]:
# Create campaign: 1 instance per VM, latest solvers only

vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-highmem-8",  # NOTE: increased to highmem!
    timeout_seconds=24 * 60 * 60,  # NOTE: 24h timeout
    solvers="gurobi mosek",  # TEST
    years=[2025],  # latest solvers only, so skip creating older conda envs
)

create_benchmark_campaign(
    "20260415-pypsa-de-elec-2-1h-mosek-test",
    "pypsa-de-elec-2-1h-mosek-test",
    vm_yamls,
)

Allocated. Estimated runtime: 158.2h
  VM 00: 1 instances, 158.2h
Created directory and files in ../infrastructure/benchmarks/20260415-pypsa-de-elec-2-1h-mosek-test
Run this campaign from the infrastructure/ directory using the command:
tofu apply -var-file benchmarks/20260415-pypsa-de-elec-2-1h-mosek-test/run.tfvars -state=states/20260415-pypsa-de-elec-2-1h-mosek-test.tfstate


## Monitor runs

To view running VMs:

`gcloud compute instances list | sort | tee /dev/tty | grep benchmark-instance | grep -i running | wc -l`

To SSH into a running VM and see what's happening:

```
gcloud compute ssh projects/compute-app-427709/zones/us-central1-a/instances/benchmark-instance-pypsa-de-elec-2-1h-mosek-test-00
tail -f /var/log/startup-script.log
cat /solver-benchmark/results/benchmark_results.csv
```

# 20260415 Run pypsa-de-elec-xx-1h (2-through-20 nodes) with Mosek

In [12]:
benchmarks_df = load_benchmark_metadata()

df = benchmarks_df[benchmarks_df["Benchmark"] == "pypsa-de-elec"].copy()

# Extract nodes
nodes = df["Instance"].str.split("-").str[0].astype(int)

# Build mask
mask = (
    df["Instance"].str.endswith("-1h")  # only 1h
    & nodes.between(2, 20)  # between 2 and 20
    & (nodes % 2 == 0)  # even only
)

benchs_to_run = df[mask]

In [13]:
# Create campaign: 1 instance per VM, latest solvers only

vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-highmem-8",  # NOTE: increased to highmem!
    timeout_seconds=24 * 60 * 60,  # NOTE: 24h timeout
    solvers="mosek",  # TEST
    years=[2025],  # latest solvers only, so skip creating older conda envs
)

create_benchmark_campaign(
    "20260415-pypsa-de-elec-2-through-20-1h-mosek",
    "pypsa-de-elec-2-through-20-1h-mosek",
    vm_yamls,
)

Allocated. Estimated runtime: 1070.7h
  VM 00: 1 instances, 1070.7h
  VM 01: 1 instances, 983.1h
  VM 02: 1 instances, 880.9h
  VM 03: 1 instances, 788.4h
  VM 04: 1 instances, 678.9h
  VM 05: 1 instances, 586.5h
  VM 06: 1 instances, 486.7h
  VM 07: 1 instances, 384.5h
  VM 08: 1 instances, 267.7h
  VM 09: 1 instances, 158.2h
Created directory and files in ../infrastructure/benchmarks/20260415-pypsa-de-elec-2-through-20-1h-mosek
Run this campaign from the infrastructure/ directory using the command:
tofu apply -var-file benchmarks/20260415-pypsa-de-elec-2-through-20-1h-mosek/run.tfvars -state=states/20260415-pypsa-de-elec-2-through-20-1h-mosek.tfstate


## Monitor runs

To view running VMs:

`gcloud compute instances list | sort | tee /dev/tty | grep benchmark-instance | grep -i running | wc -l`

To SSH into a running VM and see what's happening:

```
gcloud compute ssh projects/compute-app-427709/zones/us-central1-a/instances/benchmark-instance-pypsa-de-elec-2-through-20-1h-mosek-xx
tail -f /var/log/startup-script.log
cat /solver-benchmark/results/benchmark_results.csv
```

# 20260415 Run pypsa-de-elec-uc with increasing temporal resolution with Mosek

In [ ]:
benchmarks_df = load_benchmark_metadata()

df = benchmarks_df[(benchmarks_df["Benchmark"] == "pypsa-de-elec-uc")].copy()

# Extract nodes
df["nodes"] = df["Instance"].map(lambda i: int(i.split("-")[0]))

# Extract resolution (e.g. "168h" -> 168)
df["resolution"] = df["Instance"].map(lambda i: int(i.split("-")[-1].replace("h", "")))

# Filter
benchs_to_run = df[
    (df["nodes"] == 2) & (df["resolution"].isin([3, 5, 8, 12, 24, 36, 48, 84, 168]))
]

benchs_to_run["Instance"].tolist()

In [ ]:
# Create campaign: 1 instance per VM, latest solvers only

vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-highmem-8",  # NOTE: increased to highmem!
    timeout_seconds=24 * 60 * 60,  # NOTE: 24h timeout
    solvers="mosek",
    years=[2025],  # latest solvers only, so skip creating older conda envs
)

create_benchmark_campaign(
    "20260415-pypsa-de-elec-uc-2_nodes-scaling-mosek",
    "pypsa-de-elec-uc-2_nodes-scaling-mosek",
    vm_yamls,
)

## Inspect results

To download results:

`gsutil -m rsync -r gs://solver-benchmarks-testing/results ./results/gcp-results/`